# Demo: quantum dot temperature prediction

This notebook demonstrates the public version of the project using synthetic data only.

The original experimental files are not included in the repository because they may contain unpublished lab data and internal metadata. The synthetic table follows the same schema expected by the training pipeline.

## 1. Setup

Run this notebook from the repository root. If the package is not installed with `pip install -e .`, the cell below also adds `src/` to `sys.path`.

In [ ]:
from pathlib import Path
import sys
import time

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if SRC.exists():
    sys.path.insert(0, str(SRC))

from qdtemp.model import (
    evaluate_predictions,
    group_split,
    load_feature_table,
    make_models,
    prepare_modeling_table,
)
from qdtemp.visualization import plot_feature_importances, plot_predicted_vs_true

DATA_PATH = PROJECT_ROOT / "data" / "features_with_tau_real.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "demo_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Generate synthetic demo data

The public repository contains a script that creates a safe synthetic feature table with the same columns as the internal experimental feature table.

In [ ]:
import subprocess

subprocess.run([
    sys.executable,
    "scripts/make_synthetic_demo_data.py",
    "--out", str(DATA_PATH),
    "--n-qds", "120",
    "--spectra-per-qd", "4",
    "--seed", "42",
], check=True)

df = load_feature_table(DATA_PATH)
df.head()

## 3. Prepare the modeling table

Repeated spectra are aggregated by `(qd_id, temperature_K)` using the median. The train/test split is group-aware by `qd_id`, so the same quantum dot cannot appear in both training and test data.

In [ ]:
d_model, feature_cols = prepare_modeling_table(df)
print("Modeling table shape:", d_model.shape)
print("Feature columns:", feature_cols)

split = group_split(d_model, feature_cols, random_state=42)
print("Train rows:", len(split.X_train))
print("Test rows:", len(split.X_test))
print("Overlap in qd_id:", set(split.groups_train).intersection(set(split.groups_test)))

## 4. Train baseline models

The notebook compares a constant baseline, a linear Ridge model and a nonlinear Random Forest model.

In [ ]:
models = make_models(feature_cols, random_state=42)
metrics = []
predictions = {}

for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(split.X_train, split.y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = model.predict(split.X_test)
    predict_time = time.perf_counter() - t0

    predictions[name] = y_pred
    metrics.append({
        "model": name,
        **evaluate_predictions(split.y_test, y_pred),
        "train_time_s": train_time,
        "predict_time_s": predict_time,
    })

metrics_df = pd.DataFrame(metrics).sort_values("RMSE_K")
metrics_df

## 5. Predicted vs true temperature

The plot below shows the Random Forest predictions on held-out quantum dots.

In [ ]:
rf_pred = predictions["RandomForest"]
pred_plot = OUTPUT_DIR / "pred_vs_true_RandomForest.png"
plot_predicted_vs_true(
    split.y_test,
    rf_pred,
    "RandomForest predicted vs true temperature",
    pred_plot,
)
display(Image(filename=str(pred_plot)))

## 6. Feature importance

For the synthetic data, the most informative features should be the linewidth and lifetime descriptors, because the generator was designed to mimic temperature-dependent optical broadening and PL dynamics.

In [ ]:
rf_model = models["RandomForest"]
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance_plot = OUTPUT_DIR / "feature_importances_RandomForest.png"
plot_feature_importances(importances, "RandomForest feature importances", importance_plot)
display(importances.to_frame("importance"))
display(Image(filename=str(importance_plot)))

## 7. Optional Dask grid search

The full command-line workflow can run a Dask-backed Random Forest grid search:

```bash
python scripts/train_temperature_model.py \
  --input data/features_with_tau_real.csv \
  --output outputs/temp_prediction \
  --use-dask-grid
```

This is kept outside the notebook by default to keep the demo fast and easy to run on any laptop.

## Summary

This notebook is a public, reproducible demonstration of the same workflow used in the private research project:

1. feature table generation,
2. cleaning and grouping,
3. leakage-aware train/test splitting by quantum dot,
4. baseline and nonlinear regression models,
5. saved figures and metrics.

The notebook does not include unpublished raw experimental measurements.